# Fine-tuning BEATs với Exponential Moving Average (EMA) trên Mixture-36l

Mô hình **BEATs** (*Audio Pre-Training with Acoustic Tokenizers* - Microsoft Research) là một Audio Transformer mạnh mẽ (~90M tham số). Khi fine-tune trên tập dữ liệu downstream phân loại âm thanh (như `mixture-36l` 36 lớp), trọng số ở các epoch cuối thường bị dao động do gradient noise và dễ overfit vào các đặc trưng cục bộ.

Phương pháp **Model EMA (Exponential Moving Average)** giải quyết vấn đề này bằng cách:
1. **Duy trì mô hình bóng (Shadow EMA Model $\theta_{\text{EMA}}$):** Sau mỗi optimizer step: $\theta_{\text{EMA}}^{(t)} = \beta(t) \theta_{\text{EMA}}^{(t-1)} + (1 - \beta(t)) \theta_{\text{train}}^{(t)}$.
2. **Decay Warmup Schedule:** Hệ số $\beta(t)$ tăng dần từ $0.90 \to 0.9995$ qua $1000$ bước đầu, giúp thích nghi nhanh ở giai đoạn đầu và làm phẳng bề mặt tối ưu (flat minima) ở các epoch sau.
3. **Dual Validation:** Đánh giá song song cả Raw Model và EMA Model trên từng epoch để theo dõi mức cải thiện.
4. **Zero Overhead:** Trọng số EMA sau huấn luyện được bóc tách thành một checkpoint BEATs chuẩn để deploy, tốc độ suy luận (latency) và FLOPS giữ nguyên 100%.

| Bước | Nội dung |
| :--- | :--- |
| 1 | Cấu hình môi trường, RunPaths, Logger, Device |
| 2 | Cấu hình thực nghiệm (`config/ema_beat.yaml`) |
| 3 | Khám phá Dataset `mixture-36l` và kiểm tra shapes |
| 4 | Nạp trọng số Pretrained BEATs |
| 5 | Dựng kiến trúc: `BEATClassifier` + `BEATsModelEMA` Wrapper |
| 6 | Minh họa luồng hoạt động: 1 batch forward, backward, optimizer step & EMA update |
| 7 | Huấn luyện và Đánh giá song song (Raw Model vs EMA Model) |
| 8 | Bảng kết quả so sánh, biểu đồ đường cong loss/macro-F1 và khoảng cách trọng số |
| 9 | Xuất checkpoint Standalone phục vụ triển khai thực tế |

`RUN_TRAINING = False` (mặc định) chạy smoke test nhỏ vào `output/test/`. Đặt `True` để chạy thật vào `output/`.

In [ ]:
# (0) Thiết lập đường dẫn import repo MLNC và gói ema_beat
import os
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "rules.MD").is_file())
PACKAGE_ROOT = ROOT / "imp" / "ema-beat"
os.chdir(ROOT)
for path in (ROOT, PACKAGE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
ROOT

## (1) Tên pipeline, chế độ chạy, đường dẫn đầu ra và thiết bị

In [ ]:
# (1) Khởi tạo RunPaths, logger và xác định thiết bị chạy
import torch
from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

PIPELINE_NAME = "ema-beat"
RUN_TRAINING = False  # False: smoke test nhanh (output/test/); True: chạy thật (output/)
RESUME_RUN_ID = None  # Đặt tên thư mục nếu muốn resume, ví dụ: 'ema-beat-2026-10-29-10-00-00'

output_dir = Path("output") if RUN_TRAINING else Path("output") / "test"
if RESUME_RUN_ID:
    run = RunPaths.from_run_id(RESUME_RUN_ID, output_dir)
else:
    run = RunPaths.new(PIPELINE_NAME if RUN_TRAINING else f"{PIPELINE_NAME}-smoke", output_dir)
run.create()

logger = setup_logger(PIPELINE_NAME, run.log_file)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"run_id: {run.run_id} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

## (2) Cấu hình EMA-BEATs và Seed

File cấu hình `config/ema_beat.yaml` bao gồm các thông số: EMA decay, warmup steps, learning rate, batch size, selection metric.

In [ ]:
# (2) Nạp cấu hình từ yaml và cố định seed tái lập
from ema_beat.config import load_config, smoke_config
from src.utils.seed import set_seed

CONFIG_PATH = PACKAGE_ROOT / "config" / "ema_beat.yaml"
config = load_config(CONFIG_PATH)
if not RUN_TRAINING:
    config = smoke_config(config)

set_seed(config.train.seed)
print("=== Cấu hình EMA-BEATs ===")
print(f"EMA Decay:           {config.ema.decay}")
print(f"EMA Warmup Steps:    {config.ema.warmup_steps}")
print(f"EMA Warmup Start:    {config.ema.warmup_decay_start}")
print(f"Selection Target:    {config.ema.selection_target}")
print(f"Selection Metric:    {config.train.selection_metric}")
print(f"Epochs:              {config.train.epochs}")
print(f"Batch size:          {config.train.batch_size}")
print(f"Learning rate:       {config.train.lr}")

## (3) Dataset Mixture-36l và DataLoaders

Tập dữ liệu gồm 36 lớp nhiễu âm thanh. Mỗi clip 4 giây được đưa vào trích xuất đặc trưng spectrogram bởi BEATs front-end.

In [ ]:
# (3) Khởi tạo DataLoaders: train, validation, validation_clean
from ema_beat.pipeline import build_loaders
from scripts.download_dataset import ensure_dataset
from scripts.load_dataset import MIXTURE_KEY

dataset_root = None
if RUN_TRAINING:
    try:
        dataset_root = ensure_dataset(MIXTURE_KEY, Path("datasets"))
    except Exception as e:
        print(f"Notice: {e}")

loaders = build_loaders(config, dataset_root=dataset_root, smoke=not RUN_TRAINING or dataset_root is None)
for split_name, loader in loaders.items():
    print(f"{split_name}: {len(loader.dataset)} mẫu ({len(loader)} batches)")

sample_batch = next(iter(loaders["train"]))
print(f"\nSample batch inspection:")
print(f"  waveform shape: {sample_batch['waveform'].shape} [Batch, 64000 samples (4s @ 16kHz)]")
print(f"  target shape:   {sample_batch['target'].shape} [Batch, 36 classes]")

## (4) & (5) Nạp BEATs Pretrained Model và Khởi tạo `BEATsModelEMA`

`BEATClassifier` (90.34M tham số) được kết hợp cùng lớp bao bọc `BEATsModelEMA` để duy trì bản sao shadow weights `requires_grad=False`.

In [ ]:
# (4) & (5) Dựng mô hình BEATs và shadow model EMA
from src.model.beat import BEATClassifier
from scripts.load_pretrain import download_pretrained
from scripts.finetune import set_trainable
from ema_beat.ema import BEATsModelEMA

model_cfg_path = ROOT / config.model_config
beats = BEATClassifier.from_yaml(model_cfg_path)

if RUN_TRAINING:
    try:
        pretrain_path = download_pretrained("beats")
        report = beats.load_pretrained(pretrain_path)
        print(f"Nạp pretrain: {pretrain_path} (missing: {len(report.missing)})")
    except Exception as e:
        print(f"Could not load pretrain: {e}")

beats = set_trainable(beats.to(device), True)

# Khởi tạo wrapper EMA
model_ema = BEATsModelEMA(
    model=beats,
    decay=config.ema.decay,
    warmup_steps=config.ema.warmup_steps,
    warmup_decay_start=config.ema.warmup_decay_start,
    update_interval=config.ema.update_interval,
    device=device,
    cpu_offload=config.ema.cpu_offload,
)

total_p = beats.num_parameters(trainable_only=False)
train_p = beats.num_parameters(trainable_only=True)
initial_dist = model_ema.weight_distance(beats)

print(f"BEATs Total Parameters:     {total_p / 1e6:.2f}M")
print(f"BEATs Trainable Parameters: {train_p / 1e6:.2f}M")
print(f"Khoảng cách trọng số ban đầu ||theta_train - theta_ema||: {initial_dist:.6f}")

## (6) Minh họa luồng hoạt động của kiến trúc (Single Batch Walk-through)

Ô code dưới đây minh họa chi tiết từng bước toán học trong 1 bước huấn luyện:
1. **Forward Pass:** Waveform $\to$ Audio Spectrogram Patch Embedding $\to$ 12 lớp Transformer Encoder $\to$ Logits 36 lớp.
2. **Loss Calculation & Backward Pass:** Tính Cross-Entropy và tính gradients.
3. **Optimizer Step:** AdamW cập nhật trọng số $\theta_{\text{train}}$. Lúc này khoảng cách $D = \|\theta_{\text{train}} - \theta_{\text{EMA}}\|$ tăng vọt.
4. **EMA Shadow Update:** Cập nhật $\theta_{\text{EMA}} \leftarrow \beta \theta_{\text{EMA}} + (1 - \beta) \theta_{\text{train}}$. Khoảng cách được thu hẹp mượt mà!

In [ ]:
# (6) Minh họa trực quan 1 bước forward, loss, backward, optimizer step và EMA update
from src.loss import CrossEntropyLoss

criterion = CrossEntropyLoss(config.train.label_smoothing)
optimizer = torch.optim.AdamW(beats.parameters(), lr=config.train.lr)

waveforms = sample_batch["waveform"].to(device)
labels = sample_batch["target"].argmax(dim=1).to(device)

beats.train()
optimizer.zero_grad()

# 1. Forward Pass
out = beats(waveforms)
loss = criterion(out.logits, labels)
print(f"[1. Forward Pass] Logits shape: {out.logits.shape} | Loss CE: {loss.item():.4f}")

# 2. Backward Pass & Optimizer Step
loss.backward()
torch.nn.utils.clip_grad_norm_(beats.parameters(), max_norm=config.train.grad_clip)
optimizer.step()
dist_after_opt = model_ema.weight_distance(beats)
print(f"[2. Optimizer Step] Trọng số model đã thay đổi. Khoảng cách L2: {dist_after_opt:.6f}")

# 3. EMA Shadow Update
decay_used = model_ema.get_decay()
model_ema.update(beats)
dist_after_ema = model_ema.weight_distance(beats)
print(f"[3. EMA Update] Đã cập nhật shadow weights với decay beta={decay_used:.4f}!")
print(f"    Khoảng cách L2 sau khi EMA cập nhật: {dist_after_ema:.6f}")

## (7) Huấn luyện và Đánh giá Song Song (EMABeatTrainer)

`EMABeatTrainer` kế thừa từ `BEATTrainer` của dự án, tự động cập nhật shadow weights sau mỗi optimizer step và thực hiện **Dual Evaluation** (đo đạc cả Raw Model và EMA Model) trên mỗi epoch.

In [ ]:
# (7) Huấn luyện với EMABeatTrainer
import time
from ema_beat.trainer import EMABeatTrainer

checkpoint_path = run.checkpoint_path("beats_ema")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
resume_path = checkpoint_path.with_name("resume.pt")

trainer = EMABeatTrainer(
    model=beats,
    config=config.train,
    ema_config=config.ema,
    device=device,
    logger=logger,
    name="beats_ema",
    checkpoint_path=checkpoint_path,
    resume_path=resume_path,
)

if resume_path.is_file():
    trainer.load_resume(resume_path)

start_time = time.time()
fit_result = trainer.fit(loaders["train"], loaders["validation"])
print(f"\nHuấn luyện xong trong {time.time() - start_time:.1f}s!")
print(f"Best {fit_result.selection_metric}: {fit_result.best_score:.4f} @ epoch {fit_result.best_epoch}")

## (8) Bảng So Sánh Kết Quả & Biểu Đồ Trực Quan

So sánh trực tiếp các chỉ số đánh giá (Accuracy, Macro-F1, mAP) giữa Raw Model và EMA Shadow Model, cùng biểu đồ diễn biến của khoảng cách trọng số qua các epoch.

In [ ]:
# (8.1) Bảng so sánh Raw Model vs EMA Model trên các tập validation
from ema_beat.evaluate import format_comparison_table
from IPython.display import Markdown, display

trainer._in_fit = False
for split_name, loader in loaders.items():
    if split_name == "train":
        continue
    raw_eval = trainer._evaluate_with(trainer.model.predict_proba, loader, desc=f"Raw - {split_name}")
    ema_eval = trainer._evaluate_with(trainer.ema.predict_proba, loader, desc=f"EMA - {split_name}")
    table_md = format_comparison_table(split_name, raw_eval, ema_eval)
    display(Markdown(table_md))

In [ ]:
# (8.2) Vẽ biểu đồ so sánh: Loss, Dual Macro-F1 Curves và Khoảng cách trọng số L2
import matplotlib.pyplot as plt
from src.utils.plots import BLUE, ORANGE

history = trainer.history
if history:
    epochs = [h.get("epoch", i + 1) for i, h in enumerate(history)]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. Training Loss
    axes[0].plot(epochs, [h.get("train_loss", 0.0) for h in history], marker="o", color="tab:blue", label="Train Loss")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # 2. Validation Macro-F1: Raw vs EMA
    axes[1].plot(epochs, [h.get("val_raw_macro_f1", 0.0) for h in history], marker="o", color=BLUE, label="Raw Model")
    axes[1].plot(epochs, [h.get("val_ema_macro_f1", 0.0) for h in history], marker="s", color=ORANGE, linewidth=2, label="EMA Shadow Model")
    axes[1].set_title("Validation Macro-F1 (Raw vs EMA)")
    axes[1].set_xlabel("Epoch")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # 3. Khoảng cách trọng số ||theta_train - theta_ema||
    axes[2].plot(epochs, [h.get("weight_distance", 0.0) for h in history], marker="^", color="purple", label="||theta - theta_ema||")
    axes[2].set_title("EMA Weight Distance")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## (9) Xuất Checkpoint Standalone Phục Vụ Triển Khai Thực Tế

Trọng số của mô hình EMA được đóng gói lại thành một `BEATClassifier` chuẩn, sẵn sàng suy luận với 0 overhead.

In [ ]:
# (9) Xuất mô hình Standalone chỉ chứa trọng số EMA
standalone_model = trainer.export_inference_model()
exported_path = checkpoint_path.with_name("beats_ema_standalone.pt")
torch.save({"model": standalone_model.state_dict(), "config": standalone_model.config}, exported_path)

print(f"Mô hình Standalone EMA đã được xuất tại: {exported_path}")
print("Mô hình hoàn toàn tương thích với pipeline suy luận gốc của BEATs!")